# Center-bearing and center-distance analysis

Load saved tracking, spikes and arena geometry; calculate CB/CD tuning and classifications; save reusable results. HD remains a reference measured from the same movement intervals. Theta has its own `theta_analysis.ipynb` and does not run here.

Run on `hhw9l84` with `~/.virtualenvs/rfmapping/bin/python`. Open the saved output in `cb_cd_plotting.ipynb`. Readers never generate missing inputs.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from EBB.center_cell_io import load_center_session, load_motive_center_session, save_center_analysis
from EBB.center_cells import CenterCellConfig, analyze_center_cells, prepare_center_behavior
from EBB.center_profiles import generate_center_tc


## Recording and output


In [ ]:
recording_root = Path("/mnt/senzailab/Kai/#Recording/m20")
date = "260918"
recording_number = 11
probe_name = "A"
phase_key = "baseline"
is_motive = False  # Explicit camera choice: True = Motive circle; False = Basler rectangle.

session_id = f"{recording_root.name}_{date}_{recording_number}"
session_dir = recording_root / date / f"{date}_{recording_number}"
config = CenterCellConfig()
n_shuffles = 100
shuffle_seed = 20261019
output_dir = session_dir / "data" / "center_cells" / f"Probe{probe_name}" / phase_key
overwrite = False  # Set True explicitly to replace this saved run, or choose another directory.


## Load and prepare behavior

Movement is speed >2.5 cm/s by default. Excluded frames and tracking gaps contribute no occupancy. CB increases clockwise: 0° forward, 90° right. CD is distance to the arena center in centimeters.


In [ ]:
loader = load_motive_center_session if is_motive else load_center_session
session_data = loader(session_dir, probe=probe_name, phase=phase_key)
behavior = prepare_center_behavior(
    session_data["times_s"], session_data["xy_cm_y_down"],
    session_data["hd_cw_deg"], session_data["arena_size_cm"],
    valid_mask=session_data["valid_mask"],
    arena_shape="circle" if is_motive else "rectangle", config=config,
)
session_summary = pd.Series({
    "good_units": len(session_data["spikes_by_unit"]),
    "duration_s": np.diff(behavior["trial_interval_s"])[0],
    "moving_time_s": behavior["exposure_s"][behavior["moving_mask"]].sum(),
    "spatial_coverage": behavior["spatial_coverage"],
    "excluded_tracking_fraction": session_data["tracking_selection"]["excluded_fraction"],
}, name=session_id)
display(session_summary)


## Calculate tuning and classifications

CB/HD strength is mean vector length; CD strength is R² from a linear fit to unsmoothed rates. The existing >100 movement-spike and >80% spatial-coverage gates are unchanged. CB/HD also require complete angular coverage. Strength and split-half stability must each exceed the 99th percentile of shuffle results pooled across eligible units in this recording. Unassessed labels remain NA, distinct from negative classifications.


In [ ]:
center_results = analyze_center_cells(
    behavior, session_data["spikes_by_unit"], n_shuffles=n_shuffles, seed=shuffle_seed,
)
center_summary = center_results["summary"]
classification_counts = pd.DataFrame.from_dict({
    feature.upper(): {
        "positive": int(center_summary[f"{feature}_is_cell"].sum()),
        "assessed": int(center_summary[f"{feature}_is_cell"].notna().sum()),
        "unassessed": int(center_summary[f"{feature}_is_cell"].isna().sum()),
    }
    for feature in ("cb", "cd", "hd")
}, orient="index")
display(classification_counts, center_summary)


## Save native tuning, classifications and comparison CSVs

`cb_tuning.tc` and `cd_tuning.tc` use the HD-style unit/count/occupancy/bin/`unit_data` structure. Their classifications retain CB/CD names and criteria. CSVs contain one unit per row and numeric bin centers; CB columns are degrees, CD columns are centimeters. Complete raw curves, split halves, shuffle evidence and provenance remain in the saved run.

The explicit CSV generator pools counts and occupancy before dividing. The 30-bin CB CSV is ready for `prep_cb_tc`; CD keeps its native distance grid and uses `prep_cd_tc`. Preparation functions work in memory and never save or rerun classification.


In [ ]:
geometry = {
    "arena_shape": behavior["arena_shape"],
    "arena_size_cm": behavior["arena_size_cm"].tolist(),
}
if is_motive:
    geometry.update(session_data["geometry"])
provenance = {
    "session_id": session_id, "probe": probe_name, "phase": phase_key,
    "is_motive": is_motive, "input_files": session_data["provenance"],
    "geometry": geometry, "tracking_selection": session_data["tracking_selection"],
    "behavior_metadata": behavior["metadata"],
}


In [ ]:
save_center_analysis(output_dir, center_results, provenance=provenance, overwrite=overwrite)
unit_prefix = f"{recording_root.name}:{date}:{probe_name}"
for feature, bins in (("cb", 30), ("cd", None)):
    generate_center_tc(
        output_dir / f"{feature}_tuning.tc", output_dir / f"{feature}_comparison.csv",
        feature=feature, bins=bins, unit_prefix=unit_prefix, overwrite=overwrite,
    )
session_summary.to_csv(output_dir / "session_qc.csv")
classification_counts.to_csv(output_dir / "classification_counts.csv")
print(f"Saved CB/CD results: {output_dir}")
